In [ ]:
# Diffusion Policy (clone gchenfc/diffusion_policy then `pip install -e .`)
from diffusion_policy.common.replay_buffer import ReplayBuffer

# GML
import json
from pathlib import Path

import load_gml
import numpy as np
import tqdm.notebook as tqdm
from load_gml import Drawing

%load_ext autoreload
%autoreload 2

In [ ]:
# Quickly visualize the structure of gml files
drawing = load_gml.get_from_blackbook(30083, verbosity=3)
print(drawing.strokes[0].shape)
with np.printoptions(precision=3, suppress=True):
    print(drawing.strokes[0][:10])
# t, x, y, ?z

In [ ]:
def compute_state(stroke):
    return stroke[:, 1:3].astype(np.float32)
def compute_action(stroke):
    return np.diff(stroke[:, 1:3], axis=0, append=stroke[-1:, 1:3]).astype(np.float32)
def compute_obs(stroke):
    return None

In [ ]:
# sorted_fnames = reversed(sorted(FOLDER.glob('*.json'), key=lambda x: int(x.with_suffix('').name)))
# katsu_fnames = load_gml.filter_by_application(sorted_fnames)
# len(list(katsu_fnames))
# 46391

In [ ]:
def Drawing(item):
    try:
        return load_gml.Drawing(item)
    except Exception:
        return None

In [ ]:
FOLDER = Path('data/gml')
sorted_fnames = reversed(sorted(FOLDER.glob('*.json'), key=lambda x: int(x.with_suffix('').name)))
katsu_fnames = load_gml.filter_by_application(sorted_fnames)
drawings = map(Drawing, katsu_fnames)

dataset = ReplayBuffer.create_empty_zarr()
for i, drawing in tqdm.tqdm(enumerate(drawings), total=46391):
    if drawing is None:
        continue
    for stroke in drawing.strokes:
        dataset.add_episode({
            'state': compute_state(stroke),
            'action': compute_action(stroke),
            # 'obs': compute_obs(stroke),
        })
    if i % 3000 == 0:
        dataset.save_to_path(f'data/gml_{i:06d}.zarr')

In [ ]:
dataset.save_to_path('data/gml.zarr')

In [ ]:
import matplotlib.pyplot as plt
e = 0
for i in range(100):
    s, e = e, dataset.episode_ends[i]
    plt.plot(dataset.data.state[s:e, 0], dataset.data.state[s:e, 1])
    # print(dataset.episode_starts[i], dataset.episode_ends[i], dataset.episode_lengths[i])